# Efektywne programowanie w języku Python

## Wykład 4 — programowanie funkcyjne

### Od part03 do potoku przetwarzania

Znamy funkcje jako obiekty, domknięcia i zużywanie iteratorów. Teraz łączymy małe operacje w przetwarzanie danych: **wybierz → przekształć → zagreguj**.

Rdzeń: czystość funkcji, comprehensions, `map`, `filter`, `reduce` i podstawowe `itertools`. Implementacje demonstracyjne, dodatkowe narzędzia i dekorator z parametrem są rozszerzeniami.

Przykłady wymagają Pythona 3.10+; celowe błędy są opisane i wydzielone.

## 1. Co znaczy czysta funkcja?

W omawianych przykładach wynik zależy od jawnych argumentów, a funkcja nie zmienia obserwowalnego otoczenia. Mutacja argumentu, zapis do pliku i wypisywanie tekstu są efektami ubocznymi.

**Lokalne przypisania i pętla nie wykluczają czystości.** Można obliczać wynik imperatywnie wewnątrz funkcji bez zmieniania wejścia czy stanu modułu.

In [ ]:
def sum_squares(values):
    total = 0
    for value in values:
        total += value * value
    return total

source = [1, 2, 3]
assert sum_squares(source) == 14
assert source == [1, 2, 3]
assert sum_squares(source) == sum_squares(source)
print(sum_squares(source))

### Skutek uboczny a wynik

Ta funkcja ma inny kontrakt: modyfikuje przekazaną listę. Nie można jej swobodnie zastąpić poprzednią tylko dlatego, że obie używają podobnych działań arytmetycznych.

Oddzielenie obliczeń od wejścia/wyjścia ułatwia testy, składanie funkcji i analizę zachowania. Sam wybór `lambda` czy comprehension nie gwarantuje czystości.

In [1]:
def square_in_place(values):
    for index, value in enumerate(values):
        values[index] = value * value

changed = [1, 2, 3]
assert square_in_place(changed) is None
assert changed == [1, 4, 9]
print(changed)

[1, 4, 9]


### Python jest wieloparadygmatowy

Styl funkcyjny sprzyja małym, niezależnym funkcjom i kompozycji, ale nie daje automatycznego dowodu poprawności ani gwarancji szybkości.

Haskell ma czystą semantykę funkcyjną, obliczenia nieścisłe oraz osobny model I/O. Opis „bez zmiennych i stanu” byłby nadmiernym uproszczeniem. Na tym kursie skupiamy się na praktycznych własnościach kodu Pythona.

## 2. Przekształcenie: pętla, comprehension, map

Wymaganie: lista długości nazw języków w kolejności wejścia. Każdy wariant ma zwrócić tę samą listę i zachować dane źródłowe.

Nie ma ogólnej reguły, że któryś zapis zawsze jest „lepszy”. Wybór zależy od czytelności, wymaganej reprezentacji i pomiaru.

In [2]:
languages = ["python", "perl", "java", "c++"]
lengths_loop = []
for language in languages:
    lengths_loop.append(len(language))

lengths_comprehension = [len(language) for language in languages]
lengths_map = list(map(len, languages))
print(type(map(len, languages)))
assert lengths_loop == lengths_comprehension == lengths_map == [6, 4, 4, 3]
print(lengths_map)

<class 'map'>
[6, 4, 4, 3]


### map zwraca iterator

`map(function, iterable)` wywołuje funkcję podczas pobierania kolejnych wyników. `list(map(...))` materializuje wynik; sama konstrukcja `map(...)` jeszcze tego nie robi.

Comprehension buduje listę od razu. Wyrażenie generatorowe może zachować leniwe działanie bez `map`.

In [ ]:
length_stream = map(len, languages)
assert next(length_stream) == 6
assert list(length_stream) == [4, 4, 3]
assert list(length_stream) == []
assert list(len(language) for language in languages) == lengths_map

### Kiedy wykonuje się kod i pojawiają błędy?

Przygotowanie iteratora nie oznacza wykonania wszystkich obliczeń. Wyjątek przetwarzania może pojawić się dopiero przy konsumpcji.

Poniższe `print()` celowo służy obserwacji wykonania; ta funkcja demonstracyjna nie jest czysta.

In [3]:
def observed_float(text):
    print("Przetwarzamy:", text)
    return float(text)

converted = map(observed_float, ["1.0", "bad", "3.3"])
print("Iterator gotowy; jeszcze niczego nie przeliczono")
assert next(converted) == 1.0

Iterator gotowy; jeszcze niczego nie przeliczono
Przetwarzamy: 1.0


In [4]:
# Celowy ValueError: błąd przy pobieraniu drugiego wyniku.
next(converted)

Przetwarzamy: bad


ValueError: could not convert string to float: 'bad'

In [5]:
assert list(map(float, ["1.0", "3.3", "-4.2"])) == [1.0, 3.3, -4.2]
assert list(map(float, [])) == []

## 3. Filtrowanie

Predykat rozstrzyga, czy element pozostaje w wyniku. Porównujemy ten sam warunek, kolejność i końcową reprezentację.

`filter()` jest leniwy, więc w porównaniu z listą składaną materializujemy go przez `list()`.

In [ ]:
def is_even(number):
    return number % 2 == 0

fibs = [1, 1, 2, 3, 5, 8, 13, 21, 34]
even_loop = []
for number in fibs:
    if is_even(number):
        even_loop.append(number)

even_comprehension = [number for number in fibs if is_even(number)]
even_filter = list(filter(is_even, fibs))
assert even_loop == even_comprehension == even_filter == [2, 8, 34]
print(even_filter)

### Kontrakt obejmuje przypadki brzegowe

Użycie `filter(None, values)` oznacza filtrowanie przez truthiness, nie usuwanie wyłącznie `None`. Może usunąć także zero lub pusty napis.

Wybieraj predykat na podstawie wymagania — podobny wygląd zapisu nie zapewnia równoważności.

In [ ]:
assert list(filter(is_even, [])) == []
assert list(filter(is_even, [-3, -2, -1, 0])) == [-2, 0]
values = [None, 0, "", 2]
assert list(filter(None, values)) == [2]
assert [value for value in values if value is not None] == [0, "", 2]

## 4. lambda i funkcje nazwane

`lambda parameters: expression` tworzy funkcję z jednym wyrażeniem, którego wynik jest zwracany. Może mieć skutki uboczne, jeśli samo wyrażenie je powoduje.

Gdy funkcja potrzebuje nazwy, dokumentacji lub kilku instrukcji, zwykle wybieramy `def`. Przypisanie lambdy do nazwy jest dozwolone, ale PEP 8 zaleca w tym przypadku `def`.

In [ ]:
def triple(value):
    return value * 3

assert triple(4) == 12
assert (lambda value: value > 3)(4) is True
assert list(map(lambda value: value ** 2, range(5))) == [0, 1, 4, 9, 16]
pairs = [(4, 1), (3, -2), (8, 0)]
assert list(filter(lambda pair: pair[1] > 0, pairs)) == [(4, 1)]

### Istniejąca funkcja bywa najczytelniejsza

Zamiast `key=lambda value: abs(value)` można użyć `key=abs`. Do wybierania elementu sekwencji przydaje się `operator.itemgetter`.

`min` i `max` z parametrem `key` zwracają wybrany obiekt wejściowy, nie sam obliczony klucz.

In [ ]:
from operator import itemgetter

numbers = [2, 3, -9, 6]
assert sorted(numbers, key=abs) == [2, 3, 6, -9]
people = [("lukas", 1), ("horst", 9), ("thomas", 3)]
assert min(people, key=itemgetter(1)) == ("lukas", 1)
assert max(people, key=lambda person: person[1]) == ("horst", 9)
assert numbers == [2, 3, -9, 6]
print(sorted(people, key=itemgetter(1)))

### Rozszerzenie: lambdy w pętli — przewidź wynik

Co zwrócą trzy funkcje utworzone przez `[lambda: index for index in range(3)]`?

Domknięcia odczytują powiązanie nazwy przy wywołaniu, a nie zapisują jej wartości przy tworzeniu. Wszystkie poniższe funkcje korzystają z tego samego powiązania `index` w zakresie comprehension.

In [6]:
late_functions = [lambda: index for index in range(3)]
late_results = [function() for function in late_functions]
print(late_results)
assert late_results == [2, 2, 2]

[2, 2, 2]


### Wartość domyślna obliczana przy tworzeniu funkcji

W `lambda index=index: index` parametr jest lokalny dla danej funkcji, a jego domyślna wartość jest obliczana podczas jej tworzenia. To połączenie dwóch zasad z `part03`.

Dla obiektów mutowalnych nadal zapamiętujemy referencję, a nie głęboką kopię. Alternatywą jest fabryka funkcji z osobnym zakresem dla każdej wartości.

In [ ]:
captured_functions = [lambda index=index: index for index in range(3)]
assert [function() for function in captured_functions] == [0, 1, 2]

def make_constant(value):
    return lambda: value

factory_functions = [make_constant(index) for index in range(3)]
assert [function() for function in factory_functions] == [0, 1, 2]
print([function() for function in captured_functions])

### Rozszerzenie: stabilne sortowanie — dwa kryteria

Uporządkuj rekordy najpierw według grupy, potem rosnąco według wyniku. Dwa rekordy mają oba klucze równe: czy ich kolejność się zmieni?

Klucz będący krotką wyraża oba kryteria naraz. Przy dwóch stabilnych sortowaniach zaczynamy od kryterium mniej ważnego, a kończymy najważniejszym.

In [ ]:
records = [("B", 2, "Ola"), ("A", 2, "Ada"),
           ("A", 1, "Jan"), ("A", 2, "Ewa")]
by_tuple = sorted(records, key=lambda row: (row[0], row[1]))
by_score = sorted(records, key=itemgetter(1))
by_two_passes = sorted(by_score, key=itemgetter(0))
assert by_tuple == by_two_passes == [
    ("A", 1, "Jan"), ("A", 2, "Ada"), ("A", 2, "Ewa"), ("B", 2, "Ola")
]
assert records[0] == ("B", 2, "Ola")  # sorted() nie zmienia wejścia.
print(by_tuple)

### Rozszerzenie: functools.partial

`partial()` przygotowuje funkcję z częścią argumentów już ustaloną. Pozostałe przekazujemy przy późniejszym wywołaniu.

To praktyczne rozwinięcie funkcji jako obiektów z `part03`; nie wymaga pisania własnego domknięcia dla każdego przypadku.

In [ ]:
from functools import partial

parse_binary = partial(int, base=2)
assert parse_binary("101") == 5
assert list(map(parse_binary, ["10", "11", "100"])) == [2, 3, 4]
print(parse_binary("1010"))

## 5. Agregacja: gotowe narzędzia i reduce

Zacznij od `sum`, `min`, `max`, `math.prod`, `any` lub `all`, jeśli wyrażają zadanie. Nie używaj tych nazw jako zmiennych.

Puste wejście wymaga ustalonego zachowania: `sum([])` daje 0, `prod([])` daje 1; `min` i `max` bez wartości domyślnej zgłaszają błąd.

In [ ]:
from math import prod

values = [2, 56, 3]
assert min(values) == 2 and max(values) == 56 and sum(values) == 61
assert sum([1, 4, 6], 100) == 111
assert sum([]) == 0 and prod([]) == 1
assert min([], default=None) is None
total = 0
for value in [1, 2, 3, 4]:
    total += value
assert total == sum([1, 2, 3, 4]) == 10

### reduce składa wynik od lewej

`functools.reduce(function, iterable, initial)` używa funkcji dwóch argumentów: aktualnego akumulatora i następnego elementu. Nie jest to predykat.

Dla dodawania z początkiem 0: `(((0 + 1) + 2) + 3)`. W Pythonie 3.10 wartość początkową przekazujemy pozycyjnie. `reduce` należy do biblioteki standardowej, w module `functools`, a nie do funkcji wbudowanych.

In [7]:
from functools import reduce
from operator import add, sub

assert reduce(add, [1, 2, 3, 4], 0) == 10
assert reduce(add, [], 0) == 0
assert reduce(sub, [10, 3, 2]) == 5  # (10 - 3) - 2, nie 10 - (3 - 2).
print(reduce(add, [1, 2, 3, 4], 0))

10


In [8]:
# Celowy TypeError: puste wejście bez wartości początkowej.
reduce(add, [])

TypeError: reduce() of empty iterable with no initial value

### reduce a accumulate

`reduce` zwraca jeden wynik końcowy. `itertools.accumulate` jest iteratorem kolejnych wyników pośrednich.

Wybieraj narzędzie na podstawie potrzebnego wyniku, zamiast wymuszać jedną technikę dla wszystkich zadań.

In [9]:
import itertools as it

assert list(it.accumulate([1, 2, 3, 4])) == [1, 3, 6, 10]
assert list(it.accumulate([1, 2, 3], initial=0)) == [0, 1, 3, 6]
assert list(it.accumulate([])) == []
print(list(it.accumulate([1, 2, 3, 4])))

[1, 3, 6, 10]


## 6. Pierwszość: poprawność przed zmianą zapisu

Dla liczby całkowitej $n$:

$n$ jest pierwsza $\iff n \geq 2\ \land\ \neg\exists k\in\{2,\ldots,\lfloor\sqrt{n}\rfloor\}:\ n\bmod k=0$.

Wystarczy sprawdzić dzielniki do pierwiastka: jeśli liczba złożona jest iloczynem dwóch liczb większych od 1, przynajmniej jedna nie przekracza pierwiastka.

In [ ]:
from math import isqrt

def is_prime_loop(number):
    if number < 2:
        return False
    for divisor in range(2, isqrt(number) + 1):
        if number % divisor == 0:
            return False
    return True

### Ta pętla nie ma obserwowalnego efektu ubocznego

Przyjmujemy całkowity argument; nie zmieniamy go, nie modyfikujemy stanu zewnętrznego ani nie wypisujemy tekstu. Lokalna zmienna `divisor` służy tylko obliczeniu wyniku.

Warianty poniżej mają osobne nazwy, aby można było je sprawdzić i porównać bez nadpisywania wcześniejszej definicji.

In [ ]:
def is_prime_list(number):
    if number < 2:
        return False
    divisibility = [number % divisor == 0
                    for divisor in range(2, isqrt(number) + 1)]
    return not any(divisibility)

def is_prime_any(number):
    return number >= 2 and not any(
        number % divisor == 0 for divisor in range(2, isqrt(number) + 1)
    )

def is_prime_any(number):
    return number >= 2 and not any(
        True for divisor in range(2, isqrt(number) + 1) if number % divisor == 0 
    )

In [ ]:
def is_prime_all(number):
    return number >= 2 and all(
        number % divisor != 0 for divisor in range(2, isqrt(number) + 1)
    )

prime_methods = (is_prime_loop, is_prime_list, is_prime_any, is_prime_all)
prime_cases = {-3: False, 0: False, 1: False, 2: True, 3: True,
               4: False, 9: False, 25: False, 49: False, 97: True}
for number, expected in prime_cases.items():
    assert all(method(number) == expected for method in prime_methods)
print("Warianty zgodne dla wszystkich przypadków testowych")

### Pusta kolekcja i krótkie spięcie

`any([])` daje `False`, a `all([])` daje `True`. Dlatego warunek `number >= 2` jest niezbędny — sam brak znalezionych dzielników nie wyklucza 0 i 1.

Lista składana oblicza wszystkie wyniki przed `any`. Wyrażenie generatorowe umożliwia zakończenie na pierwszym rozstrzygającym elemencie. Pętla z `return` także może zakończyć się wcześnie.

In [ ]:
assert any([]) is False and all([]) is True
assert all(method(2) is True for method in prime_methods)

def primes_below(limit):
    return [number for number in range(2, limit) if is_prime_any(number)]

assert primes_below(2) == []
assert primes_below(10) == [2, 3, 5, 7]
print(primes_below(30))

### Czego nie robić w Pythonie 3?

- `xrange` nie istnieje — używamy `range`.
- `filter` nie ma `len`; zliczanie wszystkich elementów nie jest konieczne, jeśli potrzebujemy tylko odpowiedzi logicznej.
- Skrócenie zapisu nie naprawi błędnego kontraktu ani złego zakresu danych.

Do większych zbiorów liczb można dobrać inny algorytm, np. sito. Ten przykład służy porównaniu sposobów zapisu tego samego sprawdzenia.

## 7. itertools: składanie iteratorów

To rozwinięcie leniwego przetwarzania z `part03`. Oprócz wyniku sprawdzaj, które elementy są pobierane, czy źródło się kończy i czy narzędzie coś buforuje.

Podstawowe zastosowania: łączenie źródeł, ograniczanie strumieni, fragmenty zależne od warunku i sąsiednie pary.

### chain i chain.from_iterable

`chain(a, b)` przechodzi kolejno przez dwa źródła. `chain.from_iterable(rows)` otrzymuje jedno źródło kolejnych źródeł i spłaszcza je o jeden poziom.

`chain(["abc", "def"])` nie dzieli napisów na litery — to jedno wejście zawierające dwa elementy.

In [ ]:
assert list(it.chain("abc", "def")) == list("abcdef")
assert list(it.chain(["abc", "def"])) == ["abc", "def"]
assert list(it.chain(["abc"], ["def"])) == ["abc", "def"]
rows = [[1, 2], [], [3]]
assert list(it.chain.from_iterable(rows)) == [1, 2, 3]
assert list(it.chain.from_iterable([])) == []
print(list(it.chain.from_iterable(rows)))

### count i ograniczanie strumienia

`count(start=0, step=1)` tworzy potencjalnie nieskończony ciąg. `islice` pozwala pobrać skończony fragment bez materializowania całego źródła.

Nie wywołuj `list(it.count())`. W każdej demonstracji określamy limit. Ręczne liczniki i `break` też są poprawne, ale wymagają pilnowania granic.

In [ ]:
assert list(it.islice(it.count(2), 7)) == [2, 3, 4, 5, 6, 7, 8]
assert list(it.islice(it.count(10, 2), 4)) == [10, 12, 14, 16]
print(list(it.islice(it.count(2), 7)))

### Rozszerzenie: trzy wyniki nie oznaczają trzech sprawdzeń

Ile razy trzeba wywołać predykat, aby dostać pierwsze trzy liczby podzielne przez 1000, zaczynając od 1?

`filter` i `islice` są leniwe, ale konsument musi przejść przez odrzucone elementy. Leniwość pozwala przerwać pracę, nie usuwa kosztu wyszukiwania. Licznik poniżej jest zamierzonym efektem ubocznym instrumentacji.

In [ ]:
predicate_calls = {"count": 0}

def observed_multiple(number):
    predicate_calls["count"] += 1
    return number % 1_000 == 0

multiples = filter(observed_multiple, it.count(1))
assert predicate_calls["count"] == 0
first_three = list(it.islice(multiples, 3))
assert first_three == [1_000, 2_000, 3_000]
assert predicate_calls["count"] == 3_000
print(first_three, "sprawdzeń:", predicate_calls["count"])

### Uwaga na warunek, który nigdy nie zajdzie

Nawet `next(filter(predicate, infinite_source))` może nie zakończyć pracy, jeśli źródło nie ma pasującego elementu. Limit liczby wyników nie jest limitem liczby odczytów wejścia.

W zależności od wymagania ogranicz również wejście, np. przez `islice(source, input_limit)`. Nie uruchamiamy celowo nieskończonego wyszukiwania.

### Rozszerzenie: własny generator count

To model dydaktyczny dla liczb, a nie pełna implementacja biblioteczna. Nadajemy mu osobną nazwę i sprawdzamy skończony fragment.

In [ ]:
def count_demo(start=0, step=1):
    value = start
    while True:
        yield value
        value += step

assert list(it.islice(count_demo(10, 2), 4)) == [10, 12, 14, 16]

### islice: zakres i stan źródła

Obsługiwane wywołania: `islice(source, stop)` oraz `islice(source, start, stop, step)`.

Granica końcowa jest wyłączona. Indeksy muszą być nieujemne, a krok dodatni; `stop=None` oznacza brak tej granicy. Pomijane elementy też są pobierane ze źródła.

In [ ]:
assert list(it.islice("abcdefg", 2)) == ["a", "b"]
assert list(it.islice("abcdefg", 2, 4)) == ["c", "d"]
assert list(it.islice("abcdefg", 2, None)) == list("cdefg")
assert list(it.islice("abcdefg", 0, None, 2)) == list("aceg")
source_iterator = iter(range(10))
assert list(it.islice(source_iterator, 2, 5)) == [2, 3, 4]
assert next(source_iterator) == 5

### cycle: leniwość nie oznacza braku buforowania

`cycle(iterable)` zapamiętuje pobrane elementy, aby potem je powtarzać. Dla pustego wejścia kończy działanie. Dla nieskończonego wejścia bufor może rosnąć bez końca.

Bufor przechowuje odwołania do elementów, nie głębokie kopie ich zawartości. Podajemy skończone wejście i ograniczamy wynik.

In [ ]:
assert list(it.islice(it.cycle("ABCD"), 9)) == list("ABCDABCDA")
assert list(it.cycle([])) == []
print(list(it.islice(it.cycle("ABCD"), 9)))

### Rozszerzenie: model cycle

In [ ]:
def cycle_demo(iterable):
    saved = []
    for element in iterable:
        yield element
        saved.append(element)
    while saved:
        yield from saved

assert list(it.islice(cycle_demo("AB"), 5)) == list("ABABA")
assert list(cycle_demo([])) == []

### dropwhile nie jest zwykłym filter

`dropwhile(predicate, source)` pomija początek, dopóki predykat jest prawdziwy. Po pierwszym fałszu zwraca wszystkie pozostałe elementy, bez ponownego filtrowania.

`takewhile` zwraca początek, dopóki warunek jest prawdziwy. Element, który przerwał `takewhile`, został już pobrany ze źródła.

In [ ]:
values = [1, 4, 6, 4, 1]
assert list(it.dropwhile(lambda x: x < 5, values)) == [6, 4, 1]
assert list(filter(lambda x: x >= 5, values)) == [6]
assert list(it.takewhile(lambda x: x < 5, values)) == [1, 4]
source_iterator = iter(values)
assert list(it.takewhile(lambda x: x < 5, source_iterator)) == [1, 4]
assert list(source_iterator) == [4, 1]  # Rozstrzygające 6 już pobrano.

### Rozszerzenie: model dropwhile

In [ ]:
def dropwhile_demo(predicate, iterable):
    source = iter(iterable)
    for value in source:
        if not predicate(value):
            yield value
            break
    yield from source

assert list(dropwhile_demo(lambda x: x < 5, [1, 4, 6, 4, 1])) == [6, 4, 1]
assert list(dropwhile_demo(lambda x: True, [1, 2])) == []

### tee: niezależny odczyt, wspólne dane

`tee(source, n)` zwraca iteratory z niezależnymi pozycjami odczytu. Nie tworzy głębokich kopii elementów. Nie odczytuj równolegle pierwotnego iteratora poza tym mechanizmem.

Elementy potrzebne wolniejszej gałęzi są buforowane. Duża różnica postępów może oznaczać duży koszt pamięci; jeśli i tak potrzebujesz wszystkich danych wielokrotnie, lista może być prostsza.

In [ ]:
original_iterator = iter(range(5))
first, second = it.tee(original_iterator)
assert next(first) == 0
assert next(first) == 1
assert next(second) == 0
assert list(first) == [2, 3, 4]
assert list(second) == [1, 2, 3, 4]

### pairwise jest już w bibliotece

`itertools.pairwise` jest dostępne od Pythona 3.10. Zwraca sąsiednie pary; dla zera lub jednego elementu wynik jest pusty.

Przydaje się np. do obliczania różnic między kolejnymi pomiarami.

In [ ]:
assert list(it.pairwise([])) == []
assert list(it.pairwise([1])) == []
assert list(it.pairwise([1, 2, 3])) == [(1, 2), (2, 3)]
measurements = [10, 13, 12, 20]
differences = [right - left for left, right in it.pairwise(measurements)]
assert differences == [3, -1, 8]
print(differences)

### Rozszerzenie: naprawiony przepis z tee

`next(second, None)` pozwala obsłużyć pusty iterator bez `StopIteration`. W tym przepisie wartość domyślna jest ignorowana, więc `None` jako element wejścia nie sprawia problemu.

W produkcyjnym kodzie używaj gotowego `itertools.pairwise`, jeśli wersja Pythona na to pozwala.

In [ ]:
def pairwise_demo(iterable):
    first, second = it.tee(iterable)
    next(second, None)
    return zip(first, second)

for values in ([], [1], [1, 2, 3], [None, 1]):
    assert list(pairwise_demo(values)) == list(it.pairwise(values))

### Rozszerzenie: groupby grupuje sąsiadów

Przewidź liczbę grup dla `["A", "A", "B", "A"]`. `groupby` rozpoczyna nową grupę po zmianie klucza — nie zbiera wszystkich równych wartości z całego wejścia.

Przydaje się do kolejnych serii zdarzeń. Jeśli celem jest jedna grupa na klucz, zwykle trzeba wcześniej posortować dane według tego samego klucza.

In [ ]:
events = ["A", "A", "B", "A"]
runs = [(key, list(group)) for key, group in it.groupby(events)]
assert runs == [("A", ["A", "A"]), ("B", ["B"]), ("A", ["A"])]
collected = [(key, list(group)) for key, group in it.groupby(sorted(events))]
assert collected == [("A", ["A", "A", "A"]), ("B", ["B"])]
print("Serie:", runs)
print("Po sortowaniu:", collected)

### Grupy współdzielą źródło

Grupy z `groupby` są iteratorami korzystającymi ze wspólnego źródła. Przejście do następnej grupy zużywa resztę poprzedniej. Jeśli chcesz ją zachować, zmaterializuj ją przed dalszą iteracją.

Nie zapisuj samego `list(groupby(...))` z oczekiwaniem, że każdy iterator grupy pozostanie niezależny.

In [ ]:
groups = it.groupby(events)
first_key, first_group = next(groups)
assert first_key == "A" and next(first_group) == "A"
second_key, second_group = next(groups)
assert second_key == "B"
assert list(first_group) == []  # Reszta pierwszej grupy została pominięta.
assert list(second_group) == ["B"]

### Rozszerzenie: combinations

`combinations(iterable, r)` wybiera kombinacje długości `r` według pozycji elementów. Kolejność wyników wynika z kolejności wejścia; narzędzie nie sortuje dowolnych danych za nas.

Równe wartości w różnych pozycjach mogą dać takie same krotki w wyniku. Liczba kombinacji szybko rośnie — iterator nie usuwa kosztu ich późniejszej materializacji.

In [ ]:
assert list(it.combinations("ABCD", 2)) == [
    ("A", "B"), ("A", "C"), ("A", "D"),
    ("B", "C"), ("B", "D"), ("C", "D")
]
assert list(it.combinations("AAB", 2)) == [("A", "A"), ("A", "B"), ("A", "B")]
assert list(it.combinations([], 2)) == []
assert list(it.combinations([], 0)) == [()]
print(list(it.combinations(range(4), 3)))

## 8. Jeden potok, trzy równoważne rozwiązania

Dane: liczby całkowite. Wynik: suma kwadratów liczb dodatnich. Zero i liczby ujemne odrzucamy; puste wejście daje 0. Nie zmieniamy źródła.

Każda funkcja powinna działać także dla jednokrotnego iteratora. Dla kolejnych prób trzeba utworzyć nowy iterator — raz zużyty nie jest nowymi danymi.

In [ ]:
def positive_square_sum_loop(values):
    total = 0
    for value in values:
        if value > 0:
            total += value * value
    return total

def positive_square_sum_comprehension(values):
    return sum([value * value for value in values if value > 0])

In [ ]:
def positive_square_sum_pipeline(values):
    positive = filter(lambda value: value > 0, values)
    squared = map(lambda value: value * value, positive)
    return sum(squared)

pipeline_methods = (positive_square_sum_loop, positive_square_sum_comprehension,
                    positive_square_sum_pipeline)
for method in pipeline_methods:
    assert method(iter([-2, 0, 1, 3])) == 10
    assert method(iter([])) == 0
    assert method(iter([-2, -1])) == 0

### Ćwiczenie: dołóż wariant generatorowy

1. Napisz czwartą funkcję korzystającą z wyrażenia generatorowego i `sum`.
2. Sprawdź zgodność wyników dla pustego wejścia, jednego elementu i mieszanych znaków.
3. Sprawdź brak zmiany listy źródłowej oraz działanie z `iter(list)`.
4. Uzasadnij wybór: czytelność, buforowanie i liczba przejść po danych.

Nie zakładaj, że leniwy wariant musi być najszybszy.

### Przykładowe rozwiązanie — po dyskusji

In [ ]:
def positive_square_sum_generator(values):
    return sum(value * value for value in values if value > 0)

all_methods = (*pipeline_methods, positive_square_sum_generator)
cases = [([], 0), ([1], 1), ([-2, 0, 1, 3], 10), ([-3, -2], 0)]
for values, expected in cases:
    before = values.copy()
    for method in all_methods:
        assert method(values) == expected
        assert method(iter(values)) == expected
    assert values == before
print("Wyniki i brak mutacji potwierdzone")

### Rozszerzenie: kolejność operacji a koszt

Mamy 10 liczb, ale potrzebujemy kwadratów tylko liczb parzystych. Ile razy wykona się przekształcenie w dwóch potokach?

Dla liczb całkowitych parzystość liczby i jej kwadratu jest taka sama. W tym konkretnym przypadku wolno przesunąć filtr przed przekształcenie. Funkcja obserwacyjna liczy wywołania; nie mierzymy tu czasu.

In [ ]:
transform_calls = {"count": 0}

def observed_square(number):
    transform_calls["count"] += 1
    return number * number

mapped_first = list(filter(is_even, map(observed_square, range(10))))
mapped_first_calls = transform_calls["count"]
transform_calls["count"] = 0
filtered_first = list(map(observed_square, filter(is_even, range(10))))
assert mapped_first == filtered_first == [0, 4, 16, 36, 64]
assert mapped_first_calls == 10 and transform_calls["count"] == 5
print("Wywołania:", mapped_first_calls, "kontra", transform_calls["count"])

### Nie każdą operację można przesunąć

Filtr po przekształceniu bada wynik, a filtr przed nim — dane wejściowe. Trzeba uzasadnić równoważność predykatów oraz uwzględnić wyjątki i skutki uboczne.

Dla kwadratu i warunku „wartość większa od 4” bezmyślne przesunięcie tego samego filtra zmienia wynik. Najpierw kontrakt, dopiero potem optymalizacja.

In [ ]:
inputs = [-3, -2, 0, 2, 3, 5]
after_square = [value * value for value in inputs if value * value > 4]
before_square = [value * value for value in inputs if value > 4]
assert after_square == [9, 9, 25]
assert before_square == [25]
assert after_square != before_square
print(after_square, before_square)

### Jak porównać wydajność tych wariantów?

Stosuj procedurę z `opt01` i `opt02`: zgodność wyników → ten sam zakres pracy → powtarzany pomiar → osobny pomiar pamięci.

Mierz wynik końcowy, a nie samo utworzenie `map` czy `filter`. Dla każdej próby zapewnij świeże dane, jeśli wejście jest iteratorem. Comprehension wywołujące `function(x)` też ponosi koszt tego wywołania.

CPython 3.12 wprowadził inlining comprehensions (PEP 709). Wynik starego benchmarku nie jest aktualnym rankingiem wszystkich zapisów.

## 9. Rozszerzenie: dekorator z parametrem

W `part03` dekorator przyjmował od razu funkcję. Tutaj `tagged(prefix)` jest fabryką, która zwraca dekorator. `prefix` zostaje w domknięciu.

Zachowujemy argumenty, wynik i metadane przez `wraps`. Wypisywanie jest jawnym, zamierzonym efektem ubocznym tego dekoratora.

In [ ]:
from functools import wraps

def tagged(prefix):
    def decorate(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            print(f"{prefix}: {function.__name__}")
            return function(*args, **kwargs)
        return wrapper
    return decorate

In [ ]:
@tagged("DEMO")
def scaled_sum(left, right, *, factor=1):
    """Zwróć sumę argumentów pomnożoną przez factor."""
    return (left + right) * factor

assert scaled_sum(2, 3, factor=4) == 20
assert scaled_sum(left=2, right=3) == 5
assert scaled_sum.__name__ == "scaled_sum"
assert scaled_sum.__doc__ == scaled_sum.__wrapped__.__doc__

## Podsumowanie — przewidź i uzasadnij

1. Czy pętla i lokalne przypisanie oznaczają efekt uboczny?
2. Kiedy wykonuje się funkcja przekazana do `map`?
3. Dlaczego `number >= 2` jest niezbędne w sprawdzaniu pierwszości?
4. Czym różnią się `reduce`, `accumulate` i `sum`?
5. Dlaczego `dropwhile` nie jest odpowiednikiem filtrowania wszystkich elementów?
6. Co buforują `tee` i `cycle`, mimo że zwracają iteratory?

## Dokumentacja i dalsza lektura

- [Functional Programming HOWTO](https://docs.python.org/3.10/howto/functional.html).
- [itertools: budowanie potoków iteratorów](https://docs.python.org/3.10/library/itertools.html).
- [functools: reduce, partial i wraps](https://docs.python.org/3.10/library/functools.html).
- [PEP 8 — zalecenia dotyczące lambd](https://peps.python.org/pep-0008/#programming-recommendations).
- [Python 3.12 — inlining comprehensions](https://docs.python.org/3.12/whatsnew/3.12.html#pep-709-comprehension-inlining).
- [Haskell 2010 — wprowadzenie](https://www.haskell.org/onlinereport/haskell2010/haskellch1.html).

Dokumentacja Pythona 3.10 odpowiada środowisku kursu. Nowsze API wymagają sprawdzenia wersji; rdzeń przykładów pozostaje zgodny z 3.10+.

### Materiały dodatkowe — sprawdzone 2026-10-27

- [Socratica — Map, Filter, and Reduce Functions](https://www.youtube.com/watch?v=hUes6y2b--0).
- [Socratica — Lambda Expressions & Anonymous Functions](https://www.youtube.com/watch?v=25ovCm9jKfA).
- [Let's Learn Python #15 — Nesting Functions and Decorators](https://www.youtube.com/watch?v=fVon4QaY4wo).
- [Awesome Functional Python — zestaw odnośników](https://github.com/sfermigier/awesome-functional-python).
- [Python 3 Patterns, Recipes and Idioms — dekoratory](https://python-3-patterns-idioms-test.readthedocs.io/en/latest/PythonDecorators.html).

Potwierdzono odtwarzanie trzech filmów i dostępność dwóch stron. To materiały uzupełniające; przykłady oraz wyniki historycznych pomiarów oceniaj według aktualnego kontraktu i wersji Pythona.

### Starsze odnośniki — dostępność niepotwierdzona

- [Colton Myers — Decorators: A Powerful Weapon in your Python Arsenal](https://www.youtube.com/watch?v=9oyr0mocZTg).
- [Film o dekoratorach — qG5nov7OHn0](https://www.youtube.com/watch?v=qG5nov7OHn0).
- [Film o dekoratorach — P6pR_cUNKcU](https://www.youtube.com/watch?v=P6pR_cUNKcU).
- [Film o dekoratorach — LkpHQL863mw](https://www.youtube.com/watch?v=LkpHQL863mw).
- [Stack Builders — Functional Programming in Python](https://www.stackbuilders.com/blog/functional-programming-in-python/).

Podczas przeglądu 2026-10-27 wystąpiły problemy dostępu. Nie oznaczamy odnośników jako definitywnie martwych; podstawowy materiał nie zależy od ich dostępności.